In [58]:
import torch
from torchvision import datasets
from torchvision.transforms import v2 as T
from torch.utils.data import DataLoader

In [64]:
import torchmetrics

In [66]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [53]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

In [54]:
# !wget https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz

In [55]:
train_and_valid_data = datasets.CIFAR10(
    root='data',
    train=True,
        download=True,
    transform=toTensor)
test_data = datasets.CIFAR10(
    root='data',
     train=False,
      download=True,
      transform=toTensor)

In [56]:
train_and_valid_data

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
                 ToImage()
                 ToDtype(scale=True)
           )

In [57]:
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data,
    [45_000, 5_000],
)

In [59]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [61]:
X_sample, y_sample = next(iter(train_loader))
X_sample.shape

torch.Size([64, 3, 32, 32])

In [62]:
y_sample.shape

torch.Size([64])

In [63]:
#create a DNN with 20 hidden layers of 100 nurons each

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3*32*32, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 100), nn.SiLU(),
    nn.Linear(100, 10)
)

In [65]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [79]:
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    
    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }
    
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history['train_losses'].append(mean_loss)
        history['train_metrics'].append(metric.compute().item())
        history['valid_metrics'].append(
            evaluate_tm(model, valid_loader, metric).item()
        )

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [80]:
torch.manual_seed(42)
model.to(device)
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
accuracy = torchmetrics.Accuracy(
    task="multiclass",
    num_classes = 10,
).to(device)


_ =train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, 10)

Epoch 1/10, train loss: 2.3028, train metric: 0.1003, valid metric: 0.0976
Epoch 2/10, train loss: 2.3027, train metric: 0.1003, valid metric: 0.0976
Epoch 3/10, train loss: 2.3027, train metric: 0.0989, valid metric: 0.0976
Epoch 4/10, train loss: 2.3027, train metric: 0.0978, valid metric: 0.0942
Epoch 5/10, train loss: 2.3027, train metric: 0.0990, valid metric: 0.0942
Epoch 6/10, train loss: 2.3026, train metric: 0.0980, valid metric: 0.0942
Epoch 7/10, train loss: 2.3026, train metric: 0.0994, valid metric: 0.0942
Epoch 8/10, train loss: 2.3026, train metric: 0.0979, valid metric: 0.0942
Epoch 9/10, train loss: 2.3027, train metric: 0.0991, valid metric: 0.0942
Epoch 10/10, train loss: 2.3026, train metric: 0.0988, valid metric: 0.0976
